# Downloading MGnify Study data

The MGnify API provides access to study and analyses datasets for download. On this page we demonstrate how to:

-  **Discover** what datasets are available
-  **Download** the datasets
-  **Stream** or read in the datasets

<div style="display:flex;justify-content:center"> <button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:100px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;box-shadow:0 6px 18px rgba(15,118,110,.25);cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button> </div>

This page also serves as a runnable notebook. After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder. 

---

In [ ]:
# uncomment below if colab
#!pip install mgnipy
#!pip install asyncio



## 🎯 The Goal: Retrieve taxonomic datasets of tomato rhizosphere studies

Let's request tomato rhizosphere datasets and metadata from MGnify API.

Recall the typical workflow (from [What is MGni.Py?](TODO)):

1. Start up a `mgnipy.MGnipy` client with your desired configuration

2. Search in MGnify resources using a MGnifier glass

3. Receive a MGazine of MGnify datasets

which we will follow in this notebook

## 1. and 2. `mgnipy.MGnipy().studies` 

In the below cell we take care of
- ✅ 1. set up of our MGnipy instance and
- ✅ 2.a) preparing search for a list of tomato studies using the studies-specific `MGnifier` aka `mgnipy.V2.proxies.studies.Studies`
- ✅ 2.b) retrieving details (i.e., ALL `StudyDetail`s) for every study in the list 

In [2]:
from mgnipy import MGnipy

# 1. init with default config
MG = MGnipy()

# 2.a) setup studies mgnifier (build queries)
tomato_studies = MG.studies(
    biome_lineage="root:Host-associated:Plants:Rhizosphere", 
    search="tomato"
)

# 2.b) get the study list (execute list + all detail queries)
tomato_studies.enrich_details()

# take a look at the studies details results as a pandas df
tomato_studies.to_df(expand_nested_dicts=True)

Enriching details: 100%|██████████| 17/17 [00:02<00:00,  7.39it/s]


,accession,ena_accessions,title,updated_at,biome__biome_name,biome__lineage
0,MGYS00010296,"[ERP166137, PRJEB82448]",Microbiome-mediated tolerance of wild tomato t...,2025-07-23T14:00:52.352000+00:00,Rhizosphere,root:Host-associated:Plants:Rhizosphere
1,MGYS00010297,"[SRP333165, PRJNA755742]",Tomato rhizosphere microbiome in the pot exper...,2025-07-23T11:42:02.227000+00:00,Rhizosphere,root:Host-associated:Plants:Rhizosphere
2,MGYS00006231,"[ERP139927, PRJEB55060]",EMG produced TPA metagenomics assembly of PRJN...,2026-04-30T22:27:28.963000+00:00,Rhizosphere,root:Host-associated:Plants:Rhizosphere
3,MGYS00006204,"[ERP140102, PRJEB55219]",EMG produced TPA metagenomics assembly of PRJN...,2026-04-30T22:01:19.102000+00:00,Rhizosphere,root:Host-associated:Plants:Rhizosphere
4,MGYS00006205,"[ERP140107, PRJEB55224]",EMG produced TPA metagenomics assembly of PRJN...,2026-04-30T22:01:31.666000+00:00,Rhizosphere,root:Host-associated:Plants:Rhizosphere
5,MGYS00006230,"[ERP139923, PRJEB55057]",EMG produced TPA metagenomics assembly of PRJN...,2026-04-30T22:25:40.059000+00:00,Rhizosphere,root:Host-associated:Plants:Rhizosphere
6,MGYS00006208,"[ERP140115, PRJEB55232]",EMG produced TPA metagenomics assembly of PRJN...,2026-04-30T22:01:49.848000+00:00,Rhizosphere,root:Host-associated:Plants:Rhizosphere
7,MGYS00010257,"[SRP456588, PRJNA1004080]",Combined effect of microplastics and fungicide...,2025-07-03T18:47:35.095000+00:00,Soil,root:Host-associated:Plants:Rhizosphere:Soil
8,MGYS00010258,"[SRP517058, PRJNA1127303]",Metagenomic data of rhizosphere soil during to...,2025-07-04T11:04:15.175000+00:00,Soil,root:Host-associated:Plants:Rhizosphere:Soil
9,MGYS00010324,[PRJEB95772],Metagenome assembly of PRJNA777724 data set (T...,2025-08-07T12:30:16.888000+00:00,Soil,root:Host-associated:Plants:Rhizosphere:Soil


```{note}
There can be further steps between **2.a)** and **2.b)** -- which are still happening behind the scenes!! Rather than jumping right into `.enrich_details()` these are the additional steps between **2.a)** and **2.b)**:
```

```{margin} **2.a)**
setup
```
```python
    tomato_studies = MG.studies(
        biome_lineage="root:Host-associated:Plants:Rhizosphere", 
        search="tomato"
    )
```

```{margin}
optional preview requests before populating Studies list
```
```python
    tomato_studies.explain()
```

````{margin} first populating studies list page by page (executes queries to list endpoint)
```python
# which is same as:
for _ in range(tomato_studies.num_requests):
    tomato_studies.get()
```
````
```python
    tomato_studies.bulk_fetch()
```

````{margin} **2.b)** then getting the detail for each study in the list 
```python
# which is same as:
for _ in range(len(tomato_studies)):
    tomato_studies.get_detail()
```
````
```python
    tomato_studies.enrich_details()
```

## 3. Accessing the `MGazine` of datasets

- study details have a `mgnipy.MGazine` which allow us to download and interact with study-level datasets outputed from MGnify. 

- We can use `mgnipy.MGazine` to download the datasets onto disk or read them into our notebook. 

- To access the study's mgazine use `.datasets` 

- the str representaiton of mgazine tells us the number of download files and their {alias: url}


In [3]:
# access study mgazine
MZ = tomato_studies.datasets

# print for more info 
print(MZ)

# also can view more as df 
MZ.downloads_df

MGazine containing:
- MGnify pipeline versions: ['v5']
- Number of downloads: 35
- Short descriptions: ['Complete GO annotation',
 'GO slim annotation',
 'InterPro matches',
 'Phylum level taxonomies LSU',
 'Phylum level taxonomies SSU',
 'Taxonomic assignments LSU',
 'Taxonomic assignments SSU']



,file_type,download_type,short_description,long_description,alias,download_group,file_size_bytes,index_files,url,accession,pipeline_version
0,tsv,Taxonomic analysis,Phylum level taxonomies SSU,Phylum level taxonomies SSU (TSV),ERP139927_phylum_taxonomy_abundances_SSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_ssu_rrna,None,None,None,MGYS00006231,v5
1,tsv,Taxonomic analysis,Taxonomic assignments SSU,Taxonomic assignments SSU (TSV),ERP139927_taxonomy_abundances_SSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_ssu_rrna,None,None,None,MGYS00006231,v5
2,tsv,Taxonomic analysis,Phylum level taxonomies LSU,Phylum level taxonomies LSU (TSV),ERP139927_phylum_taxonomy_abundances_LSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_lsu_rrna,None,None,None,MGYS00006231,v5
3,tsv,Taxonomic analysis,Taxonomic assignments LSU,Taxonomic assignments LSU (TSV),ERP139927_taxonomy_abundances_LSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_lsu_rrna,None,None,None,MGYS00006231,v5
4,tsv,Functional analysis,InterPro matches,InterPro matches (TSV),ERP139927_IPR_abundances_v5.0.tsv,study_summary.v5.0.functional_analysis,None,None,None,MGYS00006231,v5
5,tsv,Functional analysis,GO slim annotation,GO slim annotation,ERP139927_GO-slim_abundances_v5.0.tsv,study_summary.v5.0.functional_analysis,None,None,None,MGYS00006231,v5
6,tsv,Functional analysis,Complete GO annotation,Complete GO annotation,ERP139927_GO_abundances_v5.0.tsv,study_summary.v5.0.functional_analysis,None,None,None,MGYS00006231,v5
7,tsv,Taxonomic analysis,Phylum level taxonomies SSU,Phylum level taxonomies SSU (TSV),ERP140102_phylum_taxonomy_abundances_SSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_ssu_rrna,None,None,None,MGYS00006204,v5
8,tsv,Taxonomic analysis,Taxonomic assignments SSU,Taxonomic assignments SSU (TSV),ERP140102_taxonomy_abundances_SSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_ssu_rrna,None,None,None,MGYS00006204,v5
9,tsv,Taxonomic analysis,Phylum level taxonomies LSU,Phylum level taxonomies LSU (TSV),ERP140102_phylum_taxonomy_abundances_LSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_lsu_rrna,None,None,None,MGYS00006204,v5


### Downloading datasets

FOR ONE download file/dataset, if wanting to `.download()` or explore/read in via `.stream()` then can pass the `url` or `alias`

In [4]:
one_alias = MZ.aliases[0]
print(one_alias)
# TODO when MGnify API V2 fully rolled out as urls are currently None  
# MZ.download(to_dir="downloads", alias=one_alias) 

ERP139927_phylum_taxonomy_abundances_SSU_v5.0.tsv


also the option to `download_all()`

In [5]:
# TODO when MGnify API V2 fully rolled out as urls are currently None  
# MZ.download_all(to_dir="downloads")

### Reading in a dataset `.stream()`

`.stream() `resolves a download alias or URL and returns the appropriate streaming handler for the file type. It supports returning either a full object (when `chunksize` is `None`) or an iterator of chunks when chunksize is provided. 

Supported formats include:
- TSV/CSV — stream_pandas (pandas) or stream_polars (polars) (handles gzipped TSV/CSV).
- FASTA / GFF / BIOM — stream_fasta, stream_gff, stream_biom (scikit-bio generators).
- JSONL / NDJSON — stream_jsonl (pandas or polars).

See [Intro to MGazine page](TODO) for more information

In [6]:
# TODO when MGnify API V2 fully rolled out as urls are currently None  
# df = MZ.stream(alias=one_alias, dataframe_engine="pandas")
# df.head()

---

## Wrap Up: 

This page was a quick start demonstration of:

1. ✅ Start up a `mgnipy.MGnipy` client with your desired configuration

2. ✅ Search in MGnify resources using a MGnifier glass

3. ✅ Receive a MGazine of MGnify datasets